<a href="https://colab.research.google.com/github/rohans-19/Job-Search-AI-Agent/blob/main/jobSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install requests beautifulsoup4 pandas


In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

In [14]:
def scrape_naukri(query, location, pages=1):
    jobs = []
    # Naukri often requires a more specific URL structure or parameters
    # Based on a quick check, query parameter is 'q' and location is 'location'
    # The base URL might also differ based on region (e.g., 'https://www.naukri.com/jobs-in-india')
    # For simplicity, let's assume a generic search URL structure for now.

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }

    for page in range(pages):
        # Naukri's pagination might be different (e.g., 'pageNo' parameter)
        # Let's assume 'pageNo' starts from 1 for simplicity.
        page_num = page + 1
        url = f"https://www.naukri.com/{query.replace(' ', '-')}-jobs-in-{location.replace(' ', '-')}-{page_num}"
        # Example: https://www.naukri.com/data-analyst-jobs-in-bengaluru-1

        print(f"\nAttempting to fetch URL: {url}")

        response = requests.get(url, headers=headers)
        print(f"HTTP Status Code: {response.status_code}")
        print(f"Response content snippet: {response.text[:1000]}...") # Uncommented for debugging

        soup = BeautifulSoup(response.text, "html.parser")

        # *** IMPORTANT ***
        # The following selectors are based on common patterns and might need to be adjusted.
        # You'll need to inspect Naukri.com's live HTML to find the exact class names or tags.
        # Look for elements that consistently contain individual job listings.

        # Common selectors for job cards on Naukri might include elements with class 'jobTuple bgWhite br4 mb-8' or similar
        job_cards = soup.find_all("article", class_="jobTuple bgWhite br4 mb-8")
        if not job_cards:
            # Try a broader search if the primary selector doesn't work
            job_cards = soup.find_all("div", class_="srp_container") # Or other broader containers
        if not job_cards:
            # A very general approach, often job listings are within div with data-job-id or similar
            job_cards = soup.find_all("div", attrs={"data-job-id": True})

        print(f"Number of job cards found: {len(job_cards)}")

        for job in job_cards:
            # These selectors are also prone to change. Inspect the HTML for current values.
            title_tag = job.find("a", class_="title fw500 ellipsis") # Common for job titles
            company_tag = job.find("a", class_="subTitle hoverDetails fw500") # Common for company names
            location_tag = job.find("span", class_="location ") # Common for locations

            jobs.append({
                "title": title_tag.text.strip() if title_tag else "N/A",
                "company": company_tag.text.strip() if company_tag else "N/A",
                "location": location_tag.text.strip() if location_tag else "N/A"
            })

        time.sleep(2) # Be polite and avoid overwhelming the server
    return jobs

In [15]:
def job_chatbot():
    print("🤖 AI Career Assistant (India) - Naukri Edition")
    print("Type 'exit' to stop\n")

    while True:
        query = input("Enter job role: ")

        if query.lower() == "exit":
            print("Good luck with your job search! 🚀")
            break

        location = input("Enter location: ")

        print("\nSearching jobs on Naukri.com...\n")

        # Call the new scrape_naukri function
        jobs = scrape_naukri(query, location, pages=1)

        if len(jobs) == 0:
            print("No jobs found.")
            continue

        for i, job in enumerate(jobs[:5], 1):
            print(f"{i}. {job['title']}")
            print(f"   Company: {job['company']}")
            print(f"   Location: {job['location']}")
            print("   ---------------------------")

        print("\n")

In [ ]:
job_chatbot()

🤖 AI Career Assistant (India)
Type 'exit' to stop



In [22]:
def fetch_jobs_adzuna(query, location, app_id, app_key):

    url = "https://api.adzuna.com/v1/api/jobs/in/search/1"

    params = {
        "app_id": app_id,
        "app_key": app_key,
        "what": query,
        "where": location,
        "results_per_page": 10
    }

    response = requests.get(url, params=params)

    if response.status_code != 200:
        print("Error:", response.status_code)
        return []

    data = response.json()

    jobs = []

    for job in data["results"]:
        jobs.append({
            "title": job["title"],
            "company": job["company"]["display_name"],
            "location": job["location"]["display_name"],
            "salary_min": job.get("salary_min"),
            "salary_max": job.get("salary_max")
        })

    return jobs

In [23]:
app_id = "YOUR_APP_ID"
app_key = "YOUR_APP_KEY"

jobs = fetch_jobs_adzuna("Python Developer", "Bengaluru", app_id, app_key)

df = pd.DataFrame(jobs)
df

Error: 401


""
